Graph Neural Networks (GNNs) represent a powerful class of neural networks designed to capture the complexity of graph-structured data. They can operate over data that is represented as nodes and edges, unlike traditional neural networks that assume a sequential or grid-like data structure. GNNs aim to learn a representation of each node by aggregating information from its neighbors, thereby embedding the structural information of the graph into low-dimensional vectors. This process, known as message passing, enables GNNs to capture both the features of individual nodes and the overall topology of the graph.

Graph Neural Networks (GNNs) represent a powerful class of neural networks designed to capture the complexity of graph-structured data. They can operate over data that is represented as nodes and edges, unlike traditional neural networks that assume a sequential or grid-like data structure. GNNs aim to learn a representation of each node by aggregating information from its neighbors, thereby embedding the structural information of the graph into low-dimensional vectors. This process, known as message passing, enables GNNs to capture both the features of individual nodes and the overall topology of the graph.

https://medium.com/@mulugetas/drug-discovery-and-graph-neural-networks-gnns-a-regression-example-fc738e0f11f3

https://sciml.ai/roadmap/

In [1]:
pip install rdkit

   ---------------------------------------- 0.0/22.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/22.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/22.5 MB ? eta -:--:--
    --------------------------------------- 0.5/22.5 MB 1.4 MB/s eta 0:00:16
   - -------------------------------------- 1.0/22.5 MB 2.0 MB/s eta 0:00:11
   -- ------------------------------------- 1.6/22.5 MB 2.1 MB/s eta 0:00:10
   --- ------------------------------------ 1.8/22.5 MB 2.0 MB/s eta 0:00:11
   ---- ----------------------------------- 2.4/22.5 MB 2.0 MB/s eta 0:00:11
   ----- ---------------------------------- 2.9/22.5 MB 2.1 MB/s eta 0:00:10
   ----- ---------------------------------- 3.1/22.5 MB 2.0 MB/s eta 0:00:10
   ------ --------------------------------- 3.7/22.5 MB 2.0 MB/s eta 0:00:10
   ------- -------------------------------- 4.5/22.5 MB 2.2 MB/s eta 0:00:09
   -------- ------------------------------- 5.0/22.5 MB 2.3 MB/s eta 0:00:08
   ----------- -----

In [2]:
from pandas.plotting import table
from rdkit.Chem import Draw
from rdkit import Chem
from sklearn.metrics import r2_score
from torch_geometric.data import DataLoader
from torch_geometric.datasets import MoleculeNet
from torch_geometric.nn import GCNConv
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
from torch_geometric.utils import to_networkx
from torch.nn import Linear

In [4]:
pip install pubchempy

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pubchempy: filename=pubchempy-1.0.4-py3-none-any.whl size=13921 sha256=f90956128c02f7414f255177481d5e9277fceba43c5ecf30157cf0f7901a498c
  Stored in directory: c:\users\lenovo\appdata\local\pip\cache\wheels\78\0f\d0\080f82ce0d7fdc771401b6acac304bd2ee77d67dee34737bd6
Successfully built pubchempy
Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: Building 'pubchempy' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pubchempy'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [5]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import os
import pandas as pd
import pubchempy
import rdkit
import time
import torch
import torch.nn.functional as F 
import warnings
warnings.filterwarnings("ignore")

Lipophilicty dataset

In [6]:
dataset = MoleculeNet(root=".", name="lipo")
data = dataset[0]

Processing...
Done!


In [7]:
print('\n======== Dataset =======\n')
print("Dataset type: ", type(dataset))
print("Dataset size (graphs): ", len(dataset))
print("Dataset features: ", dataset.num_features)
print("Dataset target: ", dataset.num_classes)
print("Dataset length: ", dataset.len)
print('\n======== first sample =======\n')
print("Dataset sample: ", data)
print("Sample  nodes: ", data.num_nodes)
print("Sample  edges: ", data.num_edges)


======== Dataset =======

Dataset type:  <class 'torch_geometric.datasets.molecule_net.MoleculeNet'>
Dataset size (graphs):  4200
Dataset features:  9
Dataset target:  553
Dataset length:  <bound method InMemoryDataset.len of Lipophilicity(4200)>

======== first sample =======

Dataset sample:  Data(x=[24, 9], edge_index=[2, 54], edge_attr=[54, 3], smiles='Cn1c(CN2CCN(CC2)c3ccc(Cl)cc3)nc4ccccc14', y=[1, 1])
Sample  nodes:  24
Sample  edges:  54


first 5 nodes from the first sample molecule:

In [8]:
dataset[0].x[:5]

tensor([[6, 0, 4, 5, 3, 0, 4, 0, 0],
        [7, 0, 3, 5, 0, 0, 3, 1, 1],
        [6, 0, 3, 5, 0, 0, 3, 1, 1],
        [6, 0, 4, 5, 2, 0, 4, 0, 0],
        [7, 0, 3, 5, 0, 0, 4, 0, 1]])

The first 5 sparse matrices (COO)

In [9]:
dataset[0].edge_index.t()[:5]

tensor([[ 0,  1],
        [ 1,  0],
        [ 1,  2],
        [ 1, 23],
        [ 2,  1]])